# 01 — Одноразова збірка SLAM-інструментів
Запускається **один раз** (≈40–70 хв). Результат лягає на Google Drive у `slam_bench/builds/`:
`slam_prefix.tar.gz` (ORB-SLAM3, DSO, OpenVINS, Pangolin, драйвери), `sources.tar.gz` (DPVO, DROID-SLAM + ваги), `wheels/` (CUDA-розширення).
Після цього `02_run_benchmark.ipynb` просто розпаковує архіви — нічого не білдиться.

**Важливо:** CUDA-розширення (DPVO/DROID) прив'язані до версій Python/PyTorch/CUDA. Версії записуються в `builds/env.txt`;
тіммейту треба запускати на тому ж образі Colab (інакше — перезапустити цей ноутбук).
Кожна комірка збірки не зупиняє решту, якщо щось не збілдилось — в кінці є звіт, що саме готове.

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/slam_bench/builds/wheels', exist_ok=True)
os.makedirs('/content/drive/MyDrive/slam_bench/results', exist_ok=True)
!nvidia-smi -L; python --version; nvcc --version | tail -2
!python -c "import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda)"

Mounted at /content/drive
GPU 0: Tesla T4 (UUID: GPU-8d8a26e7-4600-25a2-529a-da28f3920a9c)
Python 3.13.16
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0
torch 2.11.0+cu130 cuda 13.0


In [3]:
%%bash
apt-get update -qq
apt-get install -y -qq build-essential cmake git pkg-config wget unzip libeigen3-dev libopencv-dev libboost-all-dev libssl-dev libglew-dev libepoxy-dev libegl1-mesa-dev libwayland-dev libxkbcommon-dev wayland-protocols libgl1-mesa-dev libsuitesparse-dev libzip-dev libgoogle-glog-dev libgflags-dev libatlas-base-dev libyaml-cpp-dev libceres-dev xvfb ffmpeg libjpeg-dev libpng-dev libtbb-dev > /dev/null
mkdir -p /opt/slam/bin /opt/slam/lib /opt/slam/share /content/build
echo done

done


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [4]:
%%writefile /content/build/orb_run.cc
// Monocular ORB-SLAM3 driver: reads <seq>/rgb.txt (TUM style), saves full-frame + keyframe trajectories (TUM format).
// usage: orb_run vocab settings seqdir outdir pace(x realtime) viewer(0/1)
#include <System.h>
#include <opencv2/core/core.hpp>
#include <opencv2/imgcodecs.hpp>
#include <chrono>
#include <fstream>
#include <iostream>
#include <sstream>
#include <thread>
#include <vector>

int main(int argc, char **argv) {
    if (argc < 7) { std::cerr << "usage: orb_run vocab settings seqdir outdir pace viewer\n"; return 1; }
    std::string seq = argv[3], out = argv[4];
    double pace = std::atof(argv[5]);
    bool viewer = std::atoi(argv[6]) != 0;
    std::vector<double> ts; std::vector<std::string> files;
    std::ifstream f(seq + "/rgb.txt"); std::string line;
    while (std::getline(f, line)) {
        if (line.empty() || line[0] == '#') continue;
        std::stringstream ss(line); double t; std::string n; ss >> t >> n;
        ts.push_back(t); files.push_back(n);
    }
    ORB_SLAM3::System SLAM(argv[1], argv[2], ORB_SLAM3::System::MONOCULAR, viewer);
    for (size_t i = 0; i < ts.size(); i++) {
        cv::Mat im = cv::imread(seq + "/" + files[i], cv::IMREAD_UNCHANGED);
        if (im.empty()) continue;
        auto t0 = std::chrono::steady_clock::now();
        SLAM.TrackMonocular(im, ts[i]);
        double el = std::chrono::duration<double>(std::chrono::steady_clock::now() - t0).count();
        double T = (i + 1 < ts.size() ? ts[i + 1] - ts[i] : 0.0) / pace;
        if (pace > 0 && el < T) std::this_thread::sleep_for(std::chrono::duration<double>(T - el));
    }
    SLAM.Shutdown();
    SLAM.SaveTrajectoryTUM(out + "/CameraTrajectory.txt");
    SLAM.SaveKeyFrameTrajectoryTUM(out + "/KeyFrameTrajectory.txt");
    return 0;
}

Writing /content/build/orb_run.cc


In [5]:
%%writefile /content/build/ov_run.cc
// Standalone (ROS-free) OpenVINS driver. Reads <seq>/rgb.txt + <seq>/imu.txt (t wx wy wz ax ay az).
// usage: ov_run estimator_config.yaml seqdir out_tum.txt image_stride
// NOTE: written against the OpenVINS v2.7 API (VioManager); small signature tweaks may be needed for other versions.
#include <fstream>
#include <iostream>
#include <memory>
#include <sstream>
#include <vector>
#include <opencv2/imgcodecs.hpp>
#include "core/VioManager.h"
#include "core/VioManagerOptions.h"
#include "state/State.h"
#include "utils/opencv_yaml_parse.h"
#include "utils/sensor_data.h"

using namespace ov_msckf;

int main(int argc, char **argv) {
    if (argc < 5) { std::cerr << "usage: ov_run cfg seqdir out stride\n"; return 1; }
    std::string seq = argv[2]; int stride = std::max(1, std::atoi(argv[4]));
    auto parser = std::make_shared<ov_core::YamlParser>(argv[1]);
    VioManagerOptions params; params.print_and_load(parser);
    auto sys = std::make_shared<VioManager>(params);

    std::vector<ov_core::ImuData> imu;
    { std::ifstream f(seq + "/imu.txt"); double t, a1, a2, a3, g1, g2, g3;
      while (f >> t >> g1 >> g2 >> g3 >> a1 >> a2 >> a3) {
          ov_core::ImuData d; d.timestamp = t; d.wm << g1, g2, g3; d.am << a1, a2, a3; imu.push_back(d); } }
    std::vector<double> ts; std::vector<std::string> files;
    { std::ifstream f(seq + "/rgb.txt"); std::string l;
      while (std::getline(f, l)) { if (l.empty() || l[0] == '#') continue;
          std::stringstream ss(l); double t; std::string n; ss >> t >> n; ts.push_back(t); files.push_back(n); } }

    std::ofstream out(argv[3]); out.precision(9); out << std::fixed;
    size_t ii = 0;
    for (size_t k = 0; k < ts.size(); k += stride) {
        while (ii < imu.size() && imu[ii].timestamp <= ts[k]) sys->feed_measurement_imu(imu[ii++]);
        cv::Mat im = cv::imread(seq + "/" + files[k], cv::IMREAD_GRAYSCALE);
        if (im.empty()) continue;
        ov_core::CameraData cam; cam.timestamp = ts[k]; cam.sensor_ids.push_back(0);
        cam.images.push_back(im); cam.masks.push_back(cv::Mat::zeros(im.rows, im.cols, CV_8UC1));
        sys->feed_measurement_camera(cam);
        if (sys->initialized()) {
            auto st = sys->get_state();
            Eigen::Matrix3d R = st->_imu->Rot().transpose();   // R_ItoG
            Eigen::Quaterniond q(R); Eigen::Vector3d p = st->_imu->pos();
            out << st->_timestamp << " " << p.x() << " " << p.y() << " " << p.z() << " "
                << q.x() << " " << q.y() << " " << q.z() << " " << q.w() << "\n";
        }
    }
    return 0;
}

Writing /content/build/ov_run.cc


In [6]:
%%writefile /opt/slam/share/run_droid.py
"""Headless DROID-SLAM driver (based on the official demo.py). Saves camera-to-world poses (N x 7: xyz qxyzw)."""
import sys, os, argparse
p = argparse.ArgumentParser()
p.add_argument('--src'); p.add_argument('--imagedir'); p.add_argument('--calib'); p.add_argument('--out')
p.add_argument('--buffer', type=int, default=1000)
a = p.parse_args()
sys.path.insert(0, os.path.join(a.src, 'droid_slam'))
import numpy as np, cv2, torch
from droid import Droid


def image_stream():
    fx, fy, cx, cy = np.loadtxt(a.calib)[:4]
    for t, f in enumerate(sorted(os.listdir(a.imagedir))):
        im = cv2.imread(os.path.join(a.imagedir, f))
        h0, w0 = im.shape[:2]
        h1, w1 = int(h0 * np.sqrt((384 * 512) / (h0 * w0))), int(w0 * np.sqrt((384 * 512) / (h0 * w0)))
        im = cv2.resize(im, (w1, h1))[:h1 - h1 % 8, :w1 - w1 % 8]
        K = torch.as_tensor([fx, fy, cx, cy]); K[0::2] *= w1 / w0; K[1::2] *= h1 / h0
        yield t, torch.as_tensor(im).permute(2, 0, 1)[None], K


class Args:
    weights = os.path.join(a.src, 'droid.pth'); buffer = a.buffer; disable_vis = True; stereo = False
    beta = 0.3; filter_thresh = 2.4; warmup = 8; keyframe_thresh = 4.0; frontend_thresh = 16.0
    frontend_window = 25; frontend_radius = 2; frontend_nms = 1; backend_thresh = 22.0
    backend_radius = 2; backend_nms = 3; upsample = False; t0 = 0; image_size = None; reconstruction_path = None


args, droid = Args(), None
torch.multiprocessing.set_start_method('spawn', force=True)
for t, image, K in image_stream():
    if droid is None:
        args.image_size = [image.shape[2], image.shape[3]]
        droid = Droid(args)
    droid.track(t, image, intrinsics=K)
traj = droid.terminate(image_stream())
np.save(a.out, np.asarray(traj))

Writing /opt/slam/share/run_droid.py


### Pangolin (потрібен ORB-SLAM3 / DSO для нативних візуалізацій)

In [7]:
%%bash
set -e
cd /content/build
git clone --recursive --depth 1 --branch v0.8 https://github.com/stevenlovegrove/Pangolin.git
cd Pangolin && mkdir -p build && cd build
cmake .. -DCMAKE_BUILD_TYPE=Release -DCMAKE_INSTALL_PREFIX=/opt/slam -DBUILD_PANGOLIN_PYTHON=OFF \
  -DBUILD_TESTS=OFF -DBUILD_EXAMPLES=OFF -DBUILD_TOOLS=OFF -DCMAKE_CXX_FLAGS="-include cstdint -include limits"
make -j$(nproc) && make install
echo PANGOLIN_OK

Submodule path 'components/pango_python/pybind11': checked out '787d2c88cafa4d07fb38c9519c485a86323cfcf4'
Submodule path 'scripts/vcpkg': checked out 'e809a42f87565e803b2178a0c11263f462d1800a'
-- The C compiler identification is GNU 13.3.0
-- The CXX compiler identification is GNU 13.3.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found Eigen: '/usr/include/eigen3'
-- libpng Found and Enabled
-- libjpeg Found and Enabled
-- libtiff Found and Enabled
-- liblz4 Found and Enabled
-- libzstd Found and Enabled
-- Found Eigen: '/usr/include/eigen3'
-- libdc1394 Found and Enabled
-- V4L Found and Enabled
-- Found PkgConfig: /usr/bi

Cloning into 'Pangolin'...
Note: switching to 'aff6883c83f3fd7e8268a9715e84266c42e2efe3'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

Submodule 'components/pango_python/pybind11' (https://github.com/pybind/pybind11.git) registered for path 'components/pango_python/pybind11'
Submodule 'scripts/vcpkg' (https://github.com/microsoft/vcpkg.git) registered for path 'scripts/vcpkg'
Cloning into '/content/build/Pangolin/components/pango_python/pybind11'...
Cloning into '/content/build/Pangolin/scripts/vcpkg'...


### ORB-SLAM3 + власний драйвер `orb_run` (повна траєкторія в TUM-форматі, без `-march=native` → переносимо між CPU)

In [17]:
%%bash
set -e

DRIVE_BUILDS="/content/drive/MyDrive/slam_bench/builds"
mkdir -p "${DRIVE_BUILDS}/bin" "${DRIVE_BUILDS}/lib" "${DRIVE_BUILDS}/share"
export CMAKE_PREFIX_PATH=/opt/slam

# 1. Очищення та клонування
cd /content/build
rm -rf ORB_SLAM3
git clone --depth 1 https://github.com/UZ-SLAMLab/ORB_SLAM3.git
cd ORB_SLAM3

# 2. Патчинг прапорців та C++ стандарту
grep -rl "march=native" . --include=CMakeLists.txt | xargs -r sed -i 's/-march=native/-march=x86-64-v2/g'
sed -i 's/c++11/c++14/g; s/c++0x/c++14/g' CMakeLists.txt Thirdparty/DBoW2/CMakeLists.txt Thirdparty/g2o/CMakeLists.txt

# 3. Додавання кастомного orb_run (якщо він є)
if [ -f /content/build/orb_run.cc ]; then
    mkdir -p drivers
    cp /content/build/orb_run.cc drivers/
    printf '\nadd_executable(orb_run drivers/orb_run.cc)\ntarget_link_libraries(orb_run ${PROJECT_NAME})\n' >> CMakeLists.txt
fi

# 4. Збірка Thirdparty (DBoW2 та g2o)
echo "Збірка Thirdparty/DBoW2..."
(cd Thirdparty/DBoW2 && mkdir -p build && cd build && cmake .. -DCMAKE_BUILD_TYPE=Release > build.log 2>&1 && make -j$(nproc) > build.log 2>&1)

echo "Збірка Thirdparty/g2o..."
(cd Thirdparty/g2o && mkdir -p build && cd build && cmake .. -DCMAKE_BUILD_TYPE=Release > build.log 2>&1 && make -j$(nproc) > build.log 2>&1)

# 5. Розпаковка словника
if [ ! -f Vocabulary/ORBvoc.txt ]; then
    echo "Розпаковуємо ORBvoc.txt..."
    cd Vocabulary && tar -xf ORBvoc.txt.tar.gz && cd ..
fi

# 6. Збірка самого ORB-SLAM3
echo "Збірка ORB-SLAM3..."
mkdir -p build && cd build
cmake .. -DCMAKE_BUILD_TYPE=Release -DENABLE_PANGO=OFF -DCMAKE_EXE_LINKER_FLAGS="-lGLdispatch -lGL"
make -j$(nproc) > build.log 2>&1 || { echo "Помилка компіляції! Перевірте build/build.log"; exit 1; }
cd ..

# 7. Копіювання бінарників і бібліотек на Drive та в system path (/opt/slam)
echo "Копіювання файлів..."
mkdir -p /opt/slam/bin /opt/slam/lib /opt/slam/share

# Шукаємо та копіюємо бінарник (orb_run або mono_tum, якщо orb_run не створювався)
TARGET_BIN="$(find build -name orb_run -type f | head -1)"
if [ -z "$TARGET_BIN" ]; then
    TARGET_BIN="$(find build -name mono_tum -type f | head -1)"
fi

if [ -n "$TARGET_BIN" ]; then
    cp "$TARGET_BIN" "${DRIVE_BUILDS}/bin/orb_run"
    cp "$TARGET_BIN" /opt/slam/bin/orb_run
fi

# Бібліотеки
cp lib/libORB_SLAM3.so "${DRIVE_BUILDS}/lib/" && cp lib/libORB_SLAM3.so /opt/slam/lib/
[ -f Thirdparty/DBoW2/lib/libDBoW2.so ] && cp Thirdparty/DBoW2/lib/libDBoW2.so "${DRIVE_BUILDS}/lib/" && cp Thirdparty/DBoW2/lib/libDBoW2.so /opt/slam/lib/
[ -f Thirdparty/g2o/lib/libg2o.so ] && cp Thirdparty/g2o/lib/libg2o.so "${DRIVE_BUILDS}/lib/" && cp Thirdparty/g2o/lib/libg2o.so /opt/slam/lib/

# Словник
cp Vocabulary/ORBvoc.txt "${DRIVE_BUILDS}/share/"
cp Vocabulary/ORBvoc.txt /opt/slam/share/

echo "--- ФІНАЛЬНА ПЕРЕВІРКА В SLAM_BENCH ---"
ls -lh "${DRIVE_BUILDS}/bin/orb_run"
ls -lh "${DRIVE_BUILDS}/lib/libORB_SLAM3.so"
ls -lh "${DRIVE_BUILDS}/share/ORBvoc.txt"
echo "ORB_OK"

Збірка Thirdparty/DBoW2...
Збірка Thirdparty/g2o...
Розпаковуємо ORBvoc.txt...
Збірка ORB-SLAM3...
-- The C compiler identification is GNU 13.3.0
-- The CXX compiler identification is GNU 13.3.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Performing Test COMPILER_SUPPORTS_CXX11
-- Performing Test COMPILER_SUPPORTS_CXX11 - Success
-- Performing Test COMPILER_SUPPORTS_CXX0X
-- Performing Test COMPILER_SUPPORTS_CXX0X - Success
-- Using flag -std=c++14.
-- Found OpenCV: /usr (found suitable version "4.6.0", minimum required is "4.4")
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success


Cloning into 'ORB_SLAM3'...
Updating files: 100% (955/955), done.
CMake Deprecation Warning at CMakeLists.txt:1 (cmake_minimum_required):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value.  Or, use the <min>...<max> syntax
  to tell CMake that the project requires at least <min> but has been updated
  to work with policies introduced by <max> or earlier.


Build type: Release
OPENCV VERSION:
4.6.0
CMake Warning at CMakeLists.txt:43 (find_package):
  By not providing "Findrealsense2.cmake" in CMAKE_MODULE_PATH this project
  has asked CMake to find a package configuration file provided by
  "realsense2", but CMake did not find one.

  Could not find a package configuration file provided by "realsense2" with
  any of the following names:

    realsense2Config.cmake
    realsense2-config.cmake

  Add the installation prefix of "realsense2" to CMAKE_PREFIX_PATH or set
  "realsense2_DIR" to a directory containing o

In [14]:

# %%bash
# set -e
# DRIVE_BUILDS="/content/drive/MyDrive/slam_bench/builds"
# mkdir -p "${DRIVE_BUILDS}/bin" "${DRIVE_BUILDS}/lib" "${DRIVE_BUILDS}/share"
# cd /content/build/ORB_SLAM3
# mkdir -p build && cd build
# cmake .. \
#     -DCMAKE_BUILD_TYPE=Release \
#     -DENABLE_PANGO=OFF \
#     -DCMAKE_EXE_LINKER_FLAGS="-lGLdispatch -lGL"
# make -j$(nproc) mono_tum > build.log 2>&1
# cd ..
# if [ ! -f Vocabulary/ORBvoc.txt ]; then
#     echo "Розпаковуємо ORBvoc.txt..."
#     cd Vocabulary && tar -xf ORBvoc.txt.tar.gz && cd ..
# fi
# cp Examples/Monocular/mono_tum "${DRIVE_BUILDS}/bin/orb_run"
# cp lib/libORB_SLAM3.so Thirdparty/DBoW2/lib/libDBoW2.so Thirdparty/g2o/lib/libg2o.so "${DRIVE_BUILDS}/lib/"
# cp Vocabulary/ORBvoc.txt "${DRIVE_BUILDS}/share/"

# echo "--- ФІНАЛЬНА ПЕРЕВІРКА В SLAM_BENCH ---"
# ls -lh "${DRIVE_BUILDS}/bin/orb_run"
# ls -lh "${DRIVE_BUILDS}/lib/libORB_SLAM3.so"
# ls -lh "${DRIVE_BUILDS}/share/ORBvoc.txt"
# echo "ORB_OK"

-- Using flag -std=c++14.
-- BUILD TYPE:Release
-- Compiling on Unix
-- Configuring done (0.1s)
-- Generating done (0.2s)
-- Build files have been written to: /content/build/ORB_SLAM3/build
[ 96%] Built target ORB_SLAM3
[100%] Building CXX object CMakeFiles/mono_tum.dir/Examples/Monocular/mono_tum.cc.o
[100%] Linking CXX executable /content/build/ORB_SLAM3/Examples/Monocular/mono_tum
[100%] Built target mono_tum
--- ФІНАЛЬНА ПЕРЕВІРКА В SLAM_BENCH ---
-rw------- 1 root root 53K Oct 25 15:01 /content/drive/MyDrive/slam_bench/builds/bin/orb_run
-rw------- 1 root root 4.9M Oct 25 15:01 /content/drive/MyDrive/slam_bench/builds/lib/libORB_SLAM3.so
-rw------- 1 root root 139M Oct 25 15:01 /content/drive/MyDrive/slam_bench/builds/share/ORBvoc.txt
ORB_OK


CMake Deprecation Warning at CMakeLists.txt:1 (cmake_minimum_required):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value.  Or, use the <min>...<max> syntax
  to tell CMake that the project requires at least <min> but has been updated
  to work with policies introduced by <max> or earlier.


Build type: Release
OPENCV VERSION:
4.6.0
CMake Warning at CMakeLists.txt:43 (find_package):
  By not providing "Findrealsense2.cmake" in CMAKE_MODULE_PATH this project
  has asked CMake to find a package configuration file provided by
  "realsense2", but CMake did not find one.

  Could not find a package configuration file provided by "realsense2" with
  any of the following names:

    realsense2Config.cmake
    realsense2-config.cmake

  Add the installation prefix of "realsense2" to CMAKE_PREFIX_PATH or set
  "realsense2_DIR" to a directory containing one of the above files.  If
  "realsense2" provides a separate deve

### DSO (патчі під OpenCV 4; якщо збірка з Pangolin падає — збирається без вʼюера)

In [10]:
# 1. Підготовка та клонування
%cd /content
!rm -rf dso
!git clone --depth 1 https://github.com/JakobEngel/dso.git
%cd dso

# 2. Системні залежності
!apt-get update -qq && apt-get install -y -qq libeigen3-dev libzip-dev libboost-all-dev libopencv-dev

# 3. Сумісність з OpenCV 4+, GCC 13 та Boost 1.83+
!sed -i 's/-march=native/-march=x86-64-v2/g' CMakeLists.txt
!grep -rl "CV_LOAD_IMAGE\|CV_BGR2GRAY" src | xargs -r sed -i 's/CV_LOAD_IMAGE_GRAYSCALE/cv::IMREAD_GRAYSCALE/g; s/CV_LOAD_IMAGE_UNCHANGED/cv::IMREAD_UNCHANGED/g; s/CV_LOAD_IMAGE_COLOR/cv::IMREAD_COLOR/g; s/CV_BGR2GRAY/cv::COLOR_BGR2GRAY/g' 2>/dev/null || true
!sed -i '1s/^/#include <boost\/bind\/placeholders.hpp>\nusing namespace boost::placeholders;\n/' src/util/IndexThreadReduce.h

# 4. Створення Headless main та зачистка Pangolin GUI
!cp src/main_dso_pangolin.cpp src/main_dso_headless.cpp
!sed -i 's/.*viewer.*/\/\/ disabled viewer/g' src/main_dso_headless.cpp
!sed -i 's/.*PangolinDSOViewer.*/\/\/ disabled viewer/g' src/main_dso_headless.cpp
!sed -i 's/.*pangolin::.*/\/\/ disabled pangolin/g' src/main_dso_headless.cpp

# 5. Виправлений CMakeLists.txt із фільтрацією GUI джерел
clean_cmake = """
cmake_minimum_required(VERSION 3.10)
project(dso)

set(CMAKE_CXX_STANDARD 14)
set(CMAKE_BUILD_TYPE Release)

set(CMAKE_RUNTIME_OUTPUT_DIRECTORY ${CMAKE_BINARY_DIR}/bin)
set(CMAKE_ARCHIVE_OUTPUT_DIRECTORY ${CMAKE_BINARY_DIR}/lib)

set(CMAKE_CXX_FLAGS "${CMAKE_CXX_FLAGS} -O3 -msse4.2 -mpclmul -DRUN_NON_PARALLEL")

find_package(OpenCV REQUIRED)
find_package(Eigen3 REQUIRED)
find_package(Boost COMPONENTS system thread REQUIRED)
find_package(PkgConfig REQUIRED)
pkg_check_modules(LIBZIP REQUIRED libzip)

include_directories(
  ${EIGEN3_INCLUDE_DIR}
  ${OpenCV_INCLUDE_DIRS}
  ${Boost_INCLUDE_DIRS}
  ${LIBZIP_INCLUDE_DIRS}
  src
  thirdparty/Sophus
)

# Знаходимо всі C++ файли в src
file(GLOB_RECURSE ALL_SRC "src/*.cpp")

# Відфільтровуємо Pangolin wrappers та main-файли з основної бібліотеки
set(DSO_CORE_SRC "")
foreach(f ${ALL_SRC})
  if(NOT f MATCHES "IOWrapper/Pangolin" AND NOT f MATCHES "main_" AND NOT f MATCHES "_dummy")
    list(APPEND DSO_CORE_SRC ${f})
  endif()
endforeach()

# Статична бібліотека dso (тільки ядро)
add_library(dso ${DSO_CORE_SRC})
target_link_libraries(dso ${OpenCV_LIBS} ${Boost_LIBRARIES} ${LIBZIP_LIBRARIES})

# Безвіконний виконання файл
add_executable(dso_run src/main_dso_headless.cpp)
target_link_libraries(dso_run dso ${OpenCV_LIBS} ${Boost_LIBRARIES} ${LIBZIP_LIBRARIES})
"""

with open("CMakeLists.txt", "w") as f:
    f.write(clean_cmake)

# 6. Збірка бінарника
!mkdir -p build_headless
%cd build_headless
!cmake ..
!make -j$(nproc) dso_run

# 7. Збереження на Google Drive
!mkdir -p /content/drive/MyDrive/slam_bench/builds/bin /opt/slam/bin
!cp bin/dso_run /content/drive/MyDrive/slam_bench/builds/bin/dso_run
!cp bin/dso_run /opt/slam/bin/dso_run
!chmod +x /content/drive/MyDrive/slam_bench/builds/bin/dso_run /opt/slam/bin/dso_run

# 8. Фінальна перевірка
!ls -lh /content/drive/MyDrive/slam_bench/builds/bin/dso_run
"""
=== DSO УСПІШНО ЗБЕРЕЖЕНО НА GOOGLE DRIVE ===
"""

/content
Cloning into 'dso'...
remote: Enumerating objects: 100, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 100 (delta 16), reused 47 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (100/100), 885.68 KiB | 23.31 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/dso
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
/content/dso/build_headless
-- The C compiler identification is GNU 13.3.0
-- The CXX compiler identification is GNU 13.3.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - s

'\n=== DSO УСПІШНО ЗБЕРЕЖЕНО НА GOOGLE DRIVE ===\n'

In [11]:
from google.colab import drive
import os

# 1. Монтуємо Google Drive
drive.mount('/content/drive')

# 2. Створюємо системну папку для бінарників
!mkdir -p /opt/slam/bin

# 3. Копіюємо (або робимо symlink) вже згенерований dso_run
!cp /content/drive/MyDrive/slam_bench/builds/bin/dso_run /opt/slam/bin/dso_run
!chmod +x /opt/slam/bin/dso_run

# 4. Перевіряємо готовність бінарника до запуску
!/opt/slam/bin/dso_run --help

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
could not parse argument "--help"!!!!
Reading Calibration from file  ... not found. Cannot operate without calibration, shutting down.


### OpenVINS (standalone, без ROS) — *best effort*
Драйвер `ov_run.cc` написаний під API OpenVINS v2.7 (`VioManager`). Якщо збірка не пройде — решта пайплайнів не постраждає
(`OpenVINS` і `DSO+OpenVINS` отримають статус NOT_AVAILABLE).

In [12]:
%%bash
set -e
cd /content/build
rm -rf open_vins
git clone --depth 1 https://github.com/rpng/open_vins.git
cd open_vins
mkdir -p /opt/slam/share/ov_template
cp config/euroc_mav/estimator_config.yaml config/euroc_mav/kalibr_imu_chain.yaml /opt/slam/share/ov_template/ 2>/dev/null || true

cp /content/build/ov_run.cc ov_msckf/src/ov_run.cc 2>/dev/null || true
if ! grep -q "ov_run" ov_msckf/CMakeLists.txt; then
    printf '\nadd_executable(ov_run src/ov_run.cc)\ntarget_link_libraries(ov_run ov_msckf_lib ${thirdparty_libraries})\n' >> ov_msckf/CMakeLists.txt
fi

# 4. Збірка
cd ov_msckf && mkdir -p build && cd build
cmake .. -DCMAKE_BUILD_TYPE=Release -DENABLE_ROS=OFF -DCMAKE_PREFIX_PATH=/opt/slam
make -j$(nproc) ov_run

# 5. Копіювання результатів у slam_bench / opt
mkdir -p /content/drive/MyDrive/slam_bench/builds/bin /content/drive/MyDrive/slam_bench/builds/lib
cp ov_run /content/drive/MyDrive/slam_bench/builds/bin/ 2>/dev/null || cp ov_run /opt/slam/bin/
cp -n $(find /content/build/open_vins -name 'libov_*.so') /content/drive/MyDrive/slam_bench/builds/lib/ 2>/dev/null || true

echo "--- ПЕРЕВІРКА OPENVINS ---"
ls -lh /content/drive/MyDrive/slam_bench/builds/bin/ov_run
echo "OV_OK"


# %%bash
# cd /content/build
# git clone --depth 1 --branch v2.7 https://github.com/rpng/open_vins.git || git clone --depth 1 https://github.com/rpng/open_vins.git
# cd open_vins
# mkdir -p /opt/slam/share/ov_template
# cp config/euroc_mav/estimator_config.yaml config/euroc_mav/kalibr_imu_chain.yaml /opt/slam/share/ov_template/
# cp /content/build/ov_run.cc ov_msckf/src/ov_run.cc
# printf '\nadd_executable(ov_run src/ov_run.cc)\ntarget_link_libraries(ov_run ov_msckf_lib ${thirdparty_libraries})\n' >> ov_msckf/CMakeLists.txt
# cd ov_msckf && mkdir -p build && cd build
# cmake .. -DCMAKE_BUILD_TYPE=Release -DENABLE_ROS=OFF -DCMAKE_PREFIX_PATH=/opt/slam && make -j$(nproc) ov_run \
#   && cp ov_run /opt/slam/bin/ && cp -n $(find /content/build/open_vins -name 'libov_*.so') /opt/slam/lib/ && echo OV_OK \
#   || echo "!! OpenVINS build FAILED (see output above) - pipeline will be skipped"

-- The C compiler identification is GNU 13.3.0
-- The CXX compiler identification is GNU 13.3.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found OpenCV: /usr (found suitable version "4.6.0", minimum required is "4")
-- Found Boost: /usr/lib/x86_64-linux-gnu/cmake/Boost-1.83.0/BoostConfig.cmake (found version "1.83.0") found components: system filesystem thread date_time
-- Found Threads: TRUE
-- Found AMD headers in: /usr/include/suitesparse
-- Found AMD library: /usr/lib/x86_64-linux-gnu/libamd.so
-- Found CAMD headers in: /usr/include/suitesparse
-- Found CAMD library: /usr/lib/x86_64-linux-gnu/libcamd.so
-- Found CCOLAMD

Cloning into 'open_vins'...
Updating files: 100% (441/441), done.
CMake Deprecation Warning at CMakeLists.txt:1 (cmake_minimum_required):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value.  Or, use the <min>...<max> syntax
  to tell CMake that the project requires at least <min> but has been updated
  to work with policies introduced by <max> or earlier.


CMake Warning (dev) at CMakeLists.txt:11 (find_package):
  Policy CMP0167 is not set: The FindBoost module is removed.  Run "cmake
  --help-policy CMP0167" for policy details.  Use the cmake_policy command to
  set the policy and suppress this warning.

This warning is for project developers.  Use -Wno-dev to suppress it.

CMake Deprecation Warning at cmake/ROS1.cmake:1 (cmake_minimum_required):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value.  Or, use the <min>...<max> syntax
  t

### DPVO + DROID-SLAM (CUDA-розширення → wheels на Drive) та ваги

In [29]:
%%bash
set -e

# 1. Системні залежності
apt-get update -qq && apt-get install -y -qq libeigen3-dev > /dev/null 2>&1

BUILD_DIR="/content/build"
DRIVE_WHEELS="/content/drive/MyDrive/slam_bench/builds/wheels"
mkdir -p "${BUILD_DIR}" "${DRIVE_WHEELS}"

pip uninstall -y ninja > /dev/null 2>&1 || true

export CFLAGS="${CFLAGS} -Wno-error=array-bounds -Wno-array-bounds"
export CXXFLAGS="${CXXFLAGS} -Wno-error=array-bounds -Wno-array-bounds"

# ----------------------------------------------------------------------
# 1. DPVO: Відновлення та точні патчі для lietorch / dispatch.h
# ----------------------------------------------------------------------
echo "=== [1/2] Налаштування DPVO ==="
cd "${BUILD_DIR}"

if [ ! -d "DPVO" ]; then
    git clone --recursive https://github.com/princeton-vl/DPVO.git
fi
cd DPVO
git submodule update --init --recursive

# Скидаємо можливі пошкоджені файли
git checkout .

# Симлінк для Eigen
mkdir -p thirdparty/eigen-3.4.0
if [ ! -d "thirdparty/eigen-3.4.0/Eigen" ]; then
    ln -s /usr/include/eigen3/Eigen thirdparty/eigen-3.4.0/Eigen
fi

# Ваги
mkdir -p models
if [ ! -f "models/dpvo.pth" ]; then
    wget -q https://princeton-vl.github.gsc.hu/DPVO/dpvo.pth -O models/dpvo.pth || \
    wget -q https://vllab.postech.ac.kr/DPVO/dpvo.pth -O models/dpvo.pth || true
fi

# 1. Патч correlation_kernel.cu
sed -i 's/fmap1\.type()/fmap1\.scalar_type()/g' dpvo/altcorr/correlation_kernel.cu || true
sed -i 's/net\.type()/net\.scalar_type()/g' dpvo/altcorr/correlation_kernel.cu || true

# 2. Виправлення dispatch.h в lietorch
if [ -f "dpvo/lietorch/include/dispatch.h" ]; then
    sed -i 's/at::ScalarType _st = the_type\.scalar_type();/at::ScalarType _st = the_type;/g' dpvo/lietorch/include/dispatch.h || true
    sed -i 's/at::ScalarType _st = ::detail::scalar_type(the_type);/at::ScalarType _st = the_type;/g' dpvo/lietorch/include/dispatch.h || true
fi

# 3. Виправлення викликів .type() у lietorch_cpu.cpp та lietorch_gpu.cu
find dpvo/lietorch -type f \( -name "*.cpp" -o -name "*.cu" \) -exec sed -i 's/\b\([a-zA-Z0-9_]*\)\.type()/\1.scalar_type()/g' {} +

# 4. Повернення device().type() назад (оскільки device має саме метод .type())
find dpvo/lietorch -type f \( -name "*.cpp" -o -name "*.cu" \) -exec sed -i 's/\.device()\.scalar_type()/.device().type()/g' {} +

# 5. Патч застарілих типів PyTorch
find . -type f \( -name "*.cu" -o -name "*.cpp" -o -name "*.h" \) -exec sed -i 's/torch::headeronly::ScalarType/c10::ScalarType/g' {} +

# 6. Фікс конфлікту EIGEN
if [ -f "dpvo/lietorch/include/common.h" ]; then
    sed -i 's/#define EIGEN_DEFAULT_DENSE_INDEX_TYPE int/\/\/ #define EIGEN_DEFAULT_DENSE_INDEX_TYPE int/g' dpvo/lietorch/include/common.h
fi

# Збірка DPVO
echo "Збірка CUDA wheel для DPVO..."
USE_NINJA=0 python setup.py bdist_wheel --dist-dir "${DRIVE_WHEELS}"
pip install -e . --no-build-isolation

# ----------------------------------------------------------------------
# 2. DROID-SLAM: Налаштування та патч
# ----------------------------------------------------------------------
echo "=== [2/2] Налаштування DROID-SLAM ==="
cd "${BUILD_DIR}"

if [ ! -d "DROID-SLAM" ]; then
    git clone --recursive https://github.com/princeton-vl/DROID-SLAM.git
fi
cd DROID-SLAM
git submodule update --init --recursive
git checkout .

# Ваги
mkdir -p tools/checkpoints
if [ ! -f "tools/checkpoints/droid.pth" ]; then
    gdown --id 1M-X2m333R3-C4q-YpA2_s912Jj--lE06 -O tools/checkpoints/droid.pth || \
    wget -q https://vllab.postech.ac.kr/DROID-SLAM/droid.pth -O tools/checkpoints/droid.pth || true
fi

# Патчі DROID-SLAM
find . -type f \( -name "*.cu" -o -name "*.cpp" -o -name "*.h" \) -exec sed -i 's/torch::headeronly::ScalarType/c10::ScalarType/g' {} +
find . -type f \( -name "*.cu" -o -name "*.cpp" \) -exec sed -i 's/\b\([a-zA-Z0-9_]*\)\.type()/\1.scalar_type()/g' {} +
find . -type f \( -name "*.cu" -o -name "*.cpp" \) -exec sed -i 's/\.device()\.scalar_type()/.device().type()/g' {} +

# Збірка DROID-SLAM
echo "Збірка CUDA wheel для DROID-SLAM..."
USE_NINJA=0 python setup.py bdist_wheel --dist-dir "${DRIVE_WHEELS}"
pip install -e . --no-build-isolation

# ----------------------------------------------------------------------
# Перевірка результатів
# ----------------------------------------------------------------------
echo "=== ПЕРЕВІРКА ВАГ ТА WHEELS ==="
ls -lh "${DRIVE_WHEELS}"

python3 -c "import torch; import dpvo; print('DPVO OK')" || echo "Помилка DPVO"
python3 -c "import torch; import droid_backends; print('DROID-SLAM OK')" || echo "Помилка DROID-SLAM"

=== [1/2] Налаштування DPVO ===
Збірка CUDA wheel для DPVO...
running bdist_wheel
running build
running build_py
running build_ext
building 'cuda_corr' extension
x86_64-linux-gnu-g++ -Wno-error=array-bounds -Wno-array-bounds -fPIC -I/usr/local/lib/python3.13/dist-packages/torch/include -I/usr/local/lib/python3.13/dist-packages/torch/include/torch/csrc/api/include -I/usr/local/cuda/include -I/usr/include/python3.13 -c dpvo/altcorr/correlation.cpp -o build/temp.linux-x86_64-cpython-313/dpvo/altcorr/correlation.o -O3 -DTORCH_API_INCLUDE_EXTENSION_H -DTORCH_EXTENSION_NAME=cuda_corr -std=c++17
/usr/local/cuda/bin/nvcc -I/usr/local/lib/python3.13/dist-packages/torch/include -I/usr/local/lib/python3.13/dist-packages/torch/include/torch/csrc/api/include -I/usr/local/cuda/include -I/usr/include/python3.13 -c dpvo/altcorr/correlation_kernel.cu -o build/temp.linux-x86_64-cpython-313/dpvo/altcorr/correlation_kernel.o -D__CUDA_NO_HALF_OPERATORS__ -D__CUDA_NO_HALF_CONVERSIONS__ -D__CUDA_NO_BFLOAT16_

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Updated 74 paths from the index
W1025 16:37:30.027000 49193 torch/utils/cpp_extension.py:680] Attempted to use ninja as the BuildExtension backend but we could not find ninja.. Falling back to using the slow distutils backend.
/content/build/DPVO/thirdparty/eigen-3.4.0/Eigen/src/Core/DenseStorage.h(223): warning #20012-D: __host__ annotation is ignored on a non-virtual function("DenseStorage") that is explicitly defaulted on its first declaration
      __attribute__((host)) __attribute__((device)) DenseStorage(const DenseStorage&) = default;
                     ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

/content/build/DPVO/thirdparty/eigen-3.4.0/Eigen/src/Core/DenseStorage.h(223): warning #20012-D: __device__ annotation is ignored on a non-virtual function("Den

# Ваги

In [34]:
%%bash
set -e

DRIVE_BUILDS="/content/drive/MyDrive/slam_bench/builds/models"
mkdir -p "${DRIVE_BUILDS}"

echo "=== Завантаження ваг DPVO ==="
wget -q --show-progress https://vllab.postech.ac.kr/DPVO/dpvo.pth -O "${DRIVE_BUILDS}/dpvo.pth" || \
wget -q --show-progress https://github.com/princeton-vl/DPVO/releases/download/v0.1/dpvo.pth -O "${DRIVE_BUILDS}/dpvo.pth" || true

echo "=== Завантаження ваг DROID-SLAM ==="
gdown --id 1M-X2m333R3-C4q-YpA2_s912Jj--lE06 -O "${DRIVE_BUILDS}/droid.pth" || \
wget -q --show-progress https://vllab.postech.ac.kr/DROID-SLAM/droid.pth -O "${DRIVE_BUILDS}/droid.pth" || true

echo -e "\n=== ПЕРЕВІРКА РОЗМІРУ ВАГ ==="
ls -lh "${DRIVE_BUILDS}"

=== Завантаження ваг DPVO ===
=== Завантаження ваг DROID-SLAM ===

=== ПЕРЕВІРКА РОЗМІРУ ВАГ ===
total 0
-rw------- 1 root root 0 Oct 25 17:15 dpvo.pth
-rw------- 1 root root 0 Oct 25 17:15 droid.pth


/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1M-X2m333R3-C4q-YpA2_s912Jj--lE06

but Gdown can't. Please check connections and permissions.


In [39]:
#Частина ваг вантажились вручну

Завантаження dpvo.pth...
Завантаження droid.pth...
--- Перевірка розміру завантажених файлів ---
total 0
-rw------- 1 root root 0 Oct 25 17:29 dpvo.pth
-rw------- 1 root root 0 Oct 25 17:29 droid.pth


### Упаковка на Drive + звіт

In [31]:
%%bash
export LD_LIBRARY_PATH=/usr/local/lib/python3.13/dist-packages/torch/lib:$LD_LIBRARY_PATH

echo "=== ПЕРЕВІРКА ІМПОРТУ З ЛІНКУВАННЯМ LIBC10 ==="
python3 -c "import torch; import dpvo; print('DPVO OK')"
python3 -c "import torch; import droid_backends; print('DROID-SLAM OK')"

=== ПЕРЕВІРКА ІМПОРТУ З ЛІНКУВАННЯМ LIBC10 ===
DPVO OK
DROID-SLAM OK


In [32]:
# %%bash
# B=/content/drive/MyDrive/slam_bench/builds
# cp /opt/slam/share/run_droid.py /content/build/ 2>/dev/null
# tar czf $B/slam_prefix.tar.gz -C /opt slam
# tar czf $B/sources.tar.gz --exclude=.git --exclude=build --exclude='*/datasets' --exclude='*.zip' -C /content/build DPVO DROID-SLAM
# (python --version; python -c "import torch;print('torch',torch.__version__,'cuda',torch.version.cuda)"; nvcc --version | tail -1; pip freeze) > $B/env.txt
# echo "---- готовність ----"
# for f in bin/orb_run bin/dso_dataset bin/ov_run share/ORBvoc.txt; do [ -e /opt/slam/$f ] && echo "OK   $f" || echo "MISSING $f"; done
# [ -f /content/build/DPVO/dpvo.pth ] && echo "OK   DPVO weights" || echo "MISSING DPVO weights"
# [ -f /content/build/DROID-SLAM/droid.pth ] && echo "OK   DROID weights" || echo "MISSING DROID weights"
# python -c "import droid_backends, lietorch; print('OK   droid_backends/lietorch')" || echo "MISSING droid_backends"
# python -c "import cuda_corr; print('OK   dpvo cuda ext')" || echo "MISSING dpvo cuda ext"
# ls -la $B $B/wheels

---- готовність ----
MISSING bin/orb_run
MISSING bin/dso_dataset
MISSING bin/ov_run
OK   share/ORBvoc.txt
MISSING DPVO weights
MISSING DROID weights
MISSING droid_backends
MISSING dpvo cuda ext
/content/drive/MyDrive/slam_bench/builds:
total 65236
drwx------ 6 root root     4096 Oct 25 17:08 .
drwx------ 4 root root     4096 Oct 25 14:21 ..
drwx------ 2 root root     4096 Oct 25 15:01 bin
-rw------- 1 root root    14458 Oct 25 17:08 env.txt
drwx------ 2 root root     4096 Oct 25 15:25 lib
drwx------ 2 root root     4096 Oct 25 15:25 share
-rw------- 1 root root 46529657 Oct 25 17:08 slam_prefix.tar.gz
-rw------- 1 root root 20231340 Oct 25 17:08 sources.tar.gz
drwx------ 2 root root     4096 Oct 25 16:55 wheels

/content/drive/MyDrive/slam_bench/builds/wheels:
total 1673
drwx------ 2 root root    4096 Oct 25 16:55 .
drwx------ 6 root root    4096 Oct 25 17:08 ..
-rw------- 1 root root 1378359 Oct 25 16:44 dpvo-0.0.0-cp313-cp313-linux_x86_64.whl
-rw------- 1 root root  325158 Oct 25 16:

Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import droid_backends, lietorch; print('OK   droid_backends/lietorch')
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ImportError: libc10.so: cannot open shared object file: No such file or directory
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import cuda_corr; print('OK   dpvo cuda ext')
    ^^^^^^^^^^^^^^^^
ImportError: libc10.so: cannot open shared object file: No such file or directory


In [33]:
%%bash
set -e

DRIVE_BUILDS="/content/drive/MyDrive/slam_bench/builds"
BUILD_DIR="/content/build"

export LD_LIBRARY_PATH=/usr/local/lib/python3.13/dist-packages/torch/lib:$LD_LIBRARY_PATH

# 1. Синхронізація ваг у Google Drive
mkdir -p "${DRIVE_BUILDS}/models"

if [ -f "${BUILD_DIR}/DPVO/models/dpvo.pth" ]; then
    cp "${BUILD_DIR}/DPVO/models/dpvo.pth" "${DRIVE_BUILDS}/models/"
fi

if [ -f "${BUILD_DIR}/DROID-SLAM/tools/checkpoints/droid.pth" ]; then
    cp "${BUILD_DIR}/DROID-SLAM/tools/checkpoints/droid.pth" "${DRIVE_BUILDS}/models/"
fi

# 2. Збереження архівів джерел та префіксів
cd "${BUILD_DIR}"
tar -czf "${DRIVE_BUILDS}/sources.tar.gz" ORB_SLAM3 DPVO DROID-SLAM 2>/dev/null || true

# 3. Підсумковий звіт
echo "=== СТАТУС ВІДНОВЛЕННЯ ТА ЗБІРКИ ==="
echo "Wheels:"
ls -lh "${DRIVE_BUILDS}/wheels/"*.whl 2>/dev/null || echo "Немає wheels"

echo -e "\nВаги (Models):"
ls -lh "${DRIVE_BUILDS}/models/" 2>/dev/null || echo "Немає ваг"

echo -e "\nПеревірка Python модулів:"
python3 -c "import torch; import dpvo; print('DPVO: OK')"
python3 -c "import torch; import droid_backends; print('DROID-SLAM: OK')"

=== СТАТУС ВІДНОВЛЕННЯ ТА ЗБІРКИ ===
Wheels:
-rw------- 1 root root 1.4M Oct 25 16:44 /content/drive/MyDrive/slam_bench/builds/wheels/dpvo-0.0.0-cp313-cp313-linux_x86_64.whl
-rw------- 1 root root 318K Oct 25 16:55 /content/drive/MyDrive/slam_bench/builds/wheels/droid_backends-0.0.0-cp313-cp313-linux_x86_64.whl

Ваги (Models):
total 0
-rw------- 1 root root 0 Oct 25 17:09 dpvo.pth
-rw------- 1 root root 0 Oct 25 17:09 droid.pth

Перевірка Python модулів:
DPVO: OK
DROID-SLAM: OK


Підготовка у потрібний формат

In [40]:
%%bash
set -e

B="/content/drive/MyDrive/slam_bench/builds"
BUILD_DIR="/content/build"

# Упаковка префікса C++ систем
cd /opt
tar -czf "$B/slam_prefix.tar.gz" slam 2>/dev/null || true

# Упаковка сирцевого коду
if [ -d "$BUILD_DIR" ]; then
    cd "$BUILD_DIR"
    tar -czf "$B/sources.tar.gz" ORB_SLAM3 DPVO DROID-SLAM 2>/dev/null || true
fi

# Фінальна перевірка вмісту папки builds
echo "=== ВМІСТ ПАПКИ BUILDS ==="
ls -lh "$B"

=== ВМІСТ ПАПКИ BUILDS ===
total 1.2G
drwx------ 2 root root 4.0K Oct 25 15:01 bin
-rw------- 1 root root  15K Oct 25 17:08 env.txt
drwx------ 2 root root 4.0K Oct 25 15:25 lib
drwx------ 2 root root 4.0K Oct 25 17:29 models
drwx------ 2 root root 4.0K Oct 25 15:25 share
-rw------- 1 root root  45M Oct 25 17:48 slam_prefix.tar.gz
-rw------- 1 root root 1.1G Oct 25 17:51 sources.tar.gz
drwx------ 2 root root 4.0K Oct 25 16:55 wheels


In [3]:
%%bash
# 1. Створюємо цільові папки
mkdir -p /opt/slam/bin
mkdir -p /content/src/DPVO
mkdir -p /content/src/DROID-SLAM

# 2. Копіюємо ваги (.pth) з Google Drive прямо за очікуваними шляхами
B_MODELS="/content/drive/MyDrive/slam_bench/builds/models"

if [ -f "$B_MODELS/dpvo.pth" ]; then
    cp -f "$B_MODELS/dpvo.pth" /content/src/DPVO/dpvo.pth
fi

if [ -f "$B_MODELS/droid.pth" ]; then
    cp -f "$B_MODELS/droid.pth" /content/src/DROID-SLAM/droid.pth
fi

# 3. Шукаємо зкомпільовані бінарники у /content/src та копіюємо їх у /opt/slam/bin
find /content/src -name "orb_run" -type f -exec cp -f {} /opt/slam/bin/ \;
find /content/src -name "dso_dataset" -type f -exec cp -f {} /opt/slam/bin/ \;
find /content/src -name "ov_run" -type f -exec cp -f {} /opt/slam/bin/ \;

chmod +x /opt/slam/bin/* 2>/dev/null || true

# 4. Перевірка наявності файлів
echo "=== ПЕРЕВІРКА ВАГ ==="
ls -lh /content/src/DPVO/dpvo.pth 2>/dev/null || echo "dpvo.pth missing"
ls -lh /content/src/DROID-SLAM/droid.pth 2>/dev/null || echo "droid.pth missing"

echo -e "\n=== ПЕРЕВІРКА /opt/slam/bin ==="
ls -lh /opt/slam/bin/

=== ПЕРЕВІРКА ВАГ ===
dpvo.pth missing
droid.pth missing

=== ПЕРЕВІРКА /opt/slam/bin ===
total 0


In [6]:
from google.colab import drive
import os
import shutil
import glob
import subprocess

# 1. Підключаємо Drive
if not os.path.exists('/content/drive/MyDrive') and not os.path.exists('/content/drive/My Drive'):
    drive.mount('/content/drive', force_remount=True)

# 2. Автоматично знаходимо точний шлях до вашої папки slam_bench
base_candidates = [
    "/content/drive/MyDrive/slam_bench",
    "/content/drive/My Drive/slam_bench",
    "/content/drive/Shareddrives/slam_bench"
]

SLAM_BENCH_DIR = None
for candidate in base_candidates:
    if os.path.exists(candidate):
        SLAM_BENCH_DIR = candidate
        break

if not SLAM_BENCH_DIR:
    raise FileNotFoundError("Не вдалося знайти папку slam_bench у Google Drive!")

print(f" Знайдено slam_bench у: {SLAM_BENCH_DIR}")

BUILDS_DIR = os.path.join(SLAM_BENCH_DIR, "builds")

# 3. Розпаковуємо C++ бінарники з slam_prefix.tar.gz у /opt
prefix_tar = os.path.join(BUILDS_DIR, "slam_prefix.tar.gz")
if os.path.exists(prefix_tar):
    print(" Розпакування slam_prefix.tar.gz у /opt...")
    os.makedirs("/opt", exist_ok=True)
    subprocess.run(f"tar -xzf '{prefix_tar}' -C /opt", shell=True, check=True)

# Синхронізуємо окремі файли з builds/bin якщо вони там є розпаковані
bin_dir = os.path.join(BUILDS_DIR, "bin")
if os.path.exists(bin_dir):
    os.makedirs("/opt/slam/bin", exist_ok=True)
    for f in os.listdir(bin_dir):
        shutil.copy2(os.path.join(bin_dir, f), os.path.join("/opt/slam/bin", f))

# Робимо аліас dso_dataset якщо файл зветься dso_run
if os.path.exists("/opt/slam/bin/dso_run") and not os.path.exists("/opt/slam/bin/dso_dataset"):
    shutil.copy2("/opt/slam/bin/dso_run", "/opt/slam/bin/dso_dataset")

# 4. Копіюємо ваги (.pth) за шляхами, які очікує перевірка
os.makedirs("/content/src/DPVO", exist_ok=True)
os.makedirs("/content/src/DROID-SLAM", exist_ok=True)

# Шукаємо dpvo.pth та droid.pth всередині папки builds
for root, dirs, files in os.walk(BUILDS_DIR):
    for file in files:
        if file == "dpvo.pth":
            shutil.copy2(os.path.join(root, file), "/content/src/DPVO/dpvo.pth")
            print(" Скопійовано dpvo.pth -> /content/src/DPVO/dpvo.pth")
        elif file == "droid.pth":
            shutil.copy2(os.path.join(root, file), "/content/src/DROID-SLAM/droid.pth")
            print(" Скопійовано droid.pth -> /content/src/DROID-SLAM/droid.pth")

# 5. Перевіряємо результат
print("\n=== РЕЗУЛЬТАТ ПЕРЕВІРКИ ===")
print("orb_run:    ", os.path.exists("/opt/slam/bin/orb_run"))
print("dso_dataset:", os.path.exists("/opt/slam/bin/dso_dataset"))
print("ov_run:     ", os.path.exists("/opt/slam/bin/ov_run"))
print("dpvo.pth:   ", os.path.exists("/content/src/DPVO/dpvo.pth"))
print("droid.pth:  ", os.path.exists("/content/src/DROID-SLAM/droid.pth"))

=== СИНХРОНІЗАЦІЯ ФАЙЛІВ З GOOGLE DRIVE ===

=== ПЕРЕВІРКА КІНЦЕВИХ ШЛЯХІВ ===
orb_run: False
dso_dataset: False
ov_run: False
dpvo.pth: False
droid.pth: False
